<a href="https://colab.research.google.com/github/Krishishah7/nlp-learning-series/blob/main/06_llm_and_fine_tuning/48_semantic_feedback_agent/semantic_feedback_agent_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install sentence-transformers

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')

In [ ]:
feedback_memory = []

In [ ]:
def classify_intent(query):
    query = query.lower()

    if "%" in query:
        return "calculation"
    elif "what is" in query:
        return "knowledge"
    else:
        return "explanation"

In [ ]:
def calculator_tool(query):
    match = re.search(r'(\d+)%.*?(\d+)', query)
    if match:
        return (float(match.group(1)) / 100) * float(match.group(2))
    return "Calculation failed"


def knowledge_tool(query):
    if "artificial intelligence" in query.lower():
        return "Artificial Intelligence is the simulation of human intelligence in machines."
    return "Knowledge not found"


def explanation_tool(query):
    return "Explanation not found"

In [ ]:
def store_feedback(query, answer):
    embedding = model.encode(query)
    feedback_memory.append({
        "query": query,
        "embedding": embedding,
        "answer": answer
    })

In [ ]:
def retrieve_feedback(query, threshold=0.7):
    if not feedback_memory:
        return None

    query_embedding = model.encode(query)

    similarities = []

    for item in feedback_memory:
        sim = cosine_similarity(
            [query_embedding],
            [item["embedding"]]
        )[0][0]
        similarities.append(sim)

    best_idx = np.argmax(similarities)

    if similarities[best_idx] >= threshold:
        return feedback_memory[best_idx]["answer"]

    return None

In [ ]:
def semantic_agent(query):

    # check semantic memory
    feedback = retrieve_feedback(query)
    if feedback:
        return {"source": "semantic_memory", "answer": feedback}

    intent = classify_intent(query)

    if intent == "calculation":
        answer = calculator_tool(query)
    elif intent == "knowledge":
        answer = knowledge_tool(query)
    else:
        answer = explanation_tool(query)

    return {"source": "tool", "answer": answer}

In [10]:
# Step 1: wrong answer
q1 = "Explain deep learning"
print(semantic_agent(q1))

# Step 2: user feedback
store_feedback(q1, "Deep Learning is a subset of machine learning using neural networks.")

# Step 3: similar query
q2 = "Explain about deep learning"
print(semantic_agent(q2))

print(semantic_agent("Tell me about deep learning"))

{'source': 'tool', 'answer': 'Explanation not found'}
{'source': 'semantic_memory', 'answer': 'Deep Learning is a subset of machine learning using neural networks.'}
{'source': 'semantic_memory', 'answer': 'Deep Learning is a subset of machine learning using neural networks.'}
